# ΔP gate 2: why the gate works, and is it robust?

`ΔP Gate on motifbreakR` found that gating motifbreakR's alleleDiff by the CAV — the within-TF percentile of
P(bound | ref) ("P gate"), or of sign(alleleDiff) × ΔP ("ΔP direction gate"), the gated score then picked per variant —
beats motifbreakR on Malinois and Kircher, while the earlier P-weighting rules on Malinois did not. Two questions:

**1. Which change makes the difference?** Two switches, one at a time, for the P gate:
- the weight: raw P (as the earlier rule A) vs its percentile within the TF;
- the order: keep motifbreakR's call (largest |alleleDiff|) and then weight it, vs weight every call and then keep the
  largest |gated score|.
Plus **choice only**: the call chosen by the gate, but predicted with its plain alleleDiff (no weighting). If this is
as good as the full gate, the gain comes from choosing the right TF, not from rescaling. Same for the ΔP direction gate
(percentile only; a raw sign × ΔP weight can be negative).

**2. Is it carried by a few TFs or regions?** Malinois: leave one TF out at AUC ≥ 0.75 and ≥ 0.80. Kircher: leave one
region out among the 4 HepG2/K562 regions, and without regions 0 and 8 (the two the old rule was tuned on).

Descriptive: no nulls here (they are in the first notebook). CPU only, about 5–15 minutes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 0. Shared functions (copied from the first gate notebook) and the rule variants

In [ ]:
WORK_DIR = "/content/drive/MyDrive/GitHub/Biological-Foundation-Model/Notebooks/EP Pair Evaluation/per_occurrence/motif_token_study"
import os, json, re, time
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score

MAL = f"{WORK_DIR}/malinois"
POOL = f"{WORK_DIR}/fcav_pool"
OUT = f"{WORK_DIR}/deltaP_gate2"
os.makedirs(OUT, exist_ok=True)
LAYER, N_PERM = 24, 200
RULES = ["ΔP gate", "ΔP gate, 0.5 + percentile", "ΔP gate, direction", "P gate (reference)"]
pd.set_option("display.width", 250)
fmt = lambda x: f"{x:+.3f}"
AUC = {}
for f in os.listdir(f"{POOL}/fits/nt500m"):
    if f.endswith(".npz") and not f.endswith(".part.npz"):
        z = np.load(f"{POOL}/fits/nt500m/{f}", allow_pickle=False)
        AUC[f[:-4]] = float(np.asarray(z["auc"])[[int(l) for l in z["layers"]].index(LAYER)])


def pear(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    if a.std() == 0 or b.std() == 0:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def auroc(lab, s):
    lab = np.asarray(lab, bool)
    return roc_auc_score(lab, s) if 0 < lab.sum() < len(lab) else np.nan


def tf_pct(x, g):
    # percentile (0..1) of x among the entries with the same g
    o = np.lexsort((x, g))
    gs = g[o]
    st = np.r_[0, np.flatnonzero(gs[1:] != gs[:-1]) + 1]
    cnt = np.diff(np.r_[st, len(gs)])
    q = np.empty(len(x)); q[o] = (np.arange(len(gs)) - np.repeat(st, cnt) + 0.5) / np.repeat(cnt, cnt)
    return q


def pick(v, grp, n):
    # per group, the value with the largest |v| (0 for a group without entries)
    out = np.zeros(n)
    if len(v):
        o = np.lexsort((-np.abs(v), grp))
        f = o[np.r_[True, grp[o][1:] != grp[o][:-1]]]
        out[grp[f]] = v[f]
    return out


def rule_scores(T, dP, pref, n):
    # T: one row per motifbreakR call -- ad (alleleDiff), tf (code), grp (variant index); dP, pref aligned with T
    ad, tf, g = T["ad"], T["tf"], T["grp"]
    q = tf_pct(np.abs(dP), tf)
    gates = {"ΔP gate": q, "ΔP gate, 0.5 + percentile": 0.5 + q,
             "ΔP gate, direction": tf_pct(np.sign(ad) * dP, tf), "P gate (reference)": tf_pct(pref, tf)}
    out = {"motifbreakR": pick(ad, g, n)}
    for r, w in gates.items():
        out[r] = pick(ad * w, g, n)
    return out


def shuffle_tf(T, rng):
    # one permutation of the calls within each TF, applied to ΔP and P together
    o = np.lexsort((rng.random(len(T["tf"])), T["tf"]))
    base = np.argsort(T["tf"], kind="stable")
    dP, pref = np.empty(len(o)), np.empty(len(o))
    dP[base], pref[base] = T["dP"][o], T["pref"][o]
    return dP, pref


def pick_idx(v, grp, n):
    # index of the entry with the largest |v| per group, -1 for a group without entries
    out = np.full(n, -1)
    if len(v):
        o = np.lexsort((-np.abs(v), grp))
        f = o[np.r_[True, grp[o][1:] != grp[o][:-1]]]
        out[grp[f]] = f
    return out


def take(v, idx):
    out = np.zeros(len(idx))
    ok = idx >= 0
    out[ok] = v[idx[ok]]
    return out


def ablation(T, n):
    ad, tf, g, dP, pref = T["ad"], T["tf"], T["grp"], T["dP"], T["pref"]
    qP, qD = tf_pct(pref, tf), tf_pct(np.sign(ad) * dP, tf)
    i_mb = pick_idx(ad, g, n)
    out = {"motifbreakR": take(ad, i_mb)}
    for name, w, raw in (("P", qP, pref), ("ΔP direction", qD, None)):
        if raw is not None:
            out[f"{name}: raw, motifbreakR's call"] = take(ad * raw, i_mb)
            out[f"{name}: raw, weight then pick"] = take(ad * raw, pick_idx(ad * raw, g, n))
        out[f"{name}: percentile, motifbreakR's call"] = take(ad * w, i_mb)
        i_g = pick_idx(ad * w, g, n)
        out[f"{name}: percentile, weight then pick (the gate)"] = take(ad * w, i_g)
        out[f"{name}: choice only (gate's call, plain alleleDiff)"] = take(ad, i_g)
        out[f"{name}: share of variants where the gate changes the call"] = np.mean((i_g != i_mb)[i_mb >= 0])
    return out


VARIANTS = ["P: raw, motifbreakR's call", "P: raw, weight then pick", "P: percentile, motifbreakR's call",
            "P: percentile, weight then pick (the gate)", "P: choice only (gate's call, plain alleleDiff)",
            "ΔP direction: percentile, motifbreakR's call", "ΔP direction: percentile, weight then pick (the gate)",
            "ΔP direction: choice only (gate's call, plain alleleDiff)"]

---
# Section 1 — Malinois

## 1.1 Load (copied)

In [ ]:
CELLS3 = ["HepG2", "K562", "SKNSH"]
TIERS = [("all TFs", 0.0), ("AUC >= 0.70", 0.70), ("AUC >= 0.75", 0.75), ("AUC >= 0.80", 0.80)]
ACTIVE_MIN, Z_CLEAR = 1.0, 2.0
V = pd.read_csv(f"{MAL}/variants.csv.gz", index_col="vi", dtype={"chrom": str}, low_memory=False)
rd = lambda d: pd.concat([pd.read_csv(f"{d}/{f}") for f in os.listdir(d) if f.endswith(".csv") and not f.endswith(".part.csv")])
H = pd.read_csv(f"{MAL}/hits.csv")
H = H[~H["leak"]].drop_duplicates(["vi", "uid"])[["vi", "uid", "motif"]].merge(rd(f"{MAL}/palt_nt500m"), on=["vi", "uid"])
H["P_ref"] = rd(f"{MAL}/p_nt500m").set_index("uid")["P"].reindex(H["uid"]).values
H["dP"] = 1 / (1 + np.exp(-H["dec_alt"])) - H["P_ref"]
BEST = (H.assign(a=H["dP"].abs()).sort_values("a", ascending=False, kind="stable").drop_duplicates(["vi", "motif"])
        [["vi", "motif", "dP"]].merge(H.groupby(["vi", "motif"])["P_ref"].max().reset_index(), on=["vi", "motif"]))
CM = pd.read_csv(f"{MAL}/calls.csv").merge(BEST, on=["vi", "motif"])
CM["auc"] = CM["motif"].map(AUC)
print(f"{len(CM):,} motifbreakR calls with a ΔP on {CM['vi'].nunique():,} variants, {CM['motif'].nunique()} TFs")

## 1.2 Which change matters? Pearson with Δ, as the difference from motifbreakR

In [ ]:
def mal_table(L):
    L = L.sort_values(["vi", "motif"]).reset_index(drop=True)
    VI = np.unique(L["vi"].values)
    T = dict(ad=L["alleleDiff"].values, tf=pd.factorize(L["motif"])[0], grp=np.searchsorted(VI, L["vi"].values),
             dP=L["dP"].values, pref=L["P_ref"].values)
    Y = {c: V.loc[VI, f"d_{c}"].values for c in CELLS3}
    CL = {c: np.abs(Y[c]) / V.loc[VI, f"se_{c}"].values >= Z_CLEAR for c in CELLS3}
    SETS = {(c, s): np.isfinite(Y[c]) & ((V.loc[VI, f"act_{c}"].values >= ACTIVE_MIN) if s == "active" else True)
            for c in CELLS3 for s in ("all", "active")}
    return T, VI, Y, CL, SETS


rows = []
for name, thr in TIERS:
    L = CM[CM["auc"] >= thr]
    T, VI, Y, CL, SETS = mal_table(L)
    A = ablation(T, len(VI))
    for (c, s), m in SETS.items():
        base = pear(A["motifbreakR"][m], Y[c][m])
        row = {"tier": name, "cell": c, "variants": s, "motifbreakR": base}
        row.update({v: pear(A[v][m], Y[c][m]) - base for v in VARIANTS})
        row["P gate changes the call in"] = A["P: share of variants where the gate changes the call"]
        rows.append(row)
AB = pd.DataFrame(rows).set_index(["tier", "cell", "variants"])
AB.to_csv(f"{OUT}/malinois_ablation.csv")
print("difference in Pearson from motifbreakR (first column: motifbreakR itself)\n")
print(AB.T.to_string(float_format=fmt))

## 1.3 Leave one TF out (AUC ≥ 0.75 and ≥ 0.80): the gain of the two gates over motifbreakR

In [ ]:
GATES = {"P gate": "P: percentile, weight then pick (the gate)",
         "ΔP direction gate": "ΔP direction: percentile, weight then pick (the gate)"}


def gains(L):
    T, VI, Y, CL, SETS = mal_table(L)
    A = ablation(T, len(VI))
    out = {}
    for (c, s), m in SETS.items():
        b = pear(A["motifbreakR"][m], Y[c][m])
        for gname, v in GATES.items():
            out[c, s, gname] = pear(A[v][m], Y[c][m]) - b
    return out


rows, worst = [], []
t0 = time.time()
for name, thr in TIERS[2:]:
    L = CM[CM["auc"] >= thr]
    full = gains(L)
    tfs = sorted(L["motif"].unique())
    loo = {m: gains(L[L["motif"] != m]) for m in tfs}
    for key, g in full.items():
        vals = np.array([loo[m][key] for m in tfs])
        c, s, gname = key
        rows.append({"tier": name, "gate": gname, "cell": c, "variants": s, "gain, all TFs": g,
                     "min over leave-one-out": vals.min(), "median": np.median(vals), "max": vals.max(),
                     "TFs whose removal leaves gain <= 0": int((vals <= 0).sum()),
                     "most influential TF": tfs[int(np.argmin(vals))]})
    print(f"{name}: {len(tfs)} TFs, leave-one-out done  [{(time.time() - t0) / 60:.1f} min]", flush=True)
LT = pd.DataFrame(rows).set_index(["tier", "gate", "cell", "variants"])
LT.to_csv(f"{OUT}/malinois_leave_one_tf_out.csv")
print()
print(LT.to_string(float_format=fmt))

---
# Section 2 — Kircher

## 2.1 Load (copied), and which change matters, per region

In [ ]:
HOME = {"HepG2", "K562"}
K_TIERS = [("all TFs", 0.0), ("AUC >= 0.75", 0.75)]
MBR_THRESH, MBR_NEUT = 0.85, 0.4
C = json.load(open(f"{WORK_DIR}/config.json"))
MWJ = json.load(open(f"{POOL}/mpra_windows.json"))
INFO = {int(k): v for k, v in MWJ["regions"].items()}
def _read_meme(path):
    out, name, rows_, ns = {}, None, [], 20
    for line in open(path):
        t = line.split()
        if line.startswith("MOTIF"):
            if name is not None and name not in out:
                out[name] = (np.array(rows_), ns)
            name, rows_, ns = (t[2] if len(t) > 2 else t[1]), [], 20
        elif line.startswith("letter-probability"):
            mm = re.search(r"nsites=\s*([0-9.]+)", line)
            ns = float(mm.group(1)) if mm else 20
        elif name is not None and len(t) == 4 and all(re.fullmatch(r"[0-9.eE+-]+", x) for x in t):
            rows_.append([float(x) for x in t])
    if name is not None and name not in out:
        out[name] = (np.array(rows_), ns)
    return out


MBR_MATS = {}
for name, (f, ns) in _read_meme(C["meme_path"]).items():
    f = f / f.sum(1, keepdims=True)
    pp = (f * ns + 0.25) / (ns + 1)
    S = pp * (pp.max(1) - pp.min(1))[:, None]
    S = np.vstack([S.T, np.zeros(len(S))]).T
    MBR_MATS[name] = (S, S[::-1][:, [3, 2, 1, 0, 4]], S[:, :4].min(1).sum(), S[:, :4].max(1).sum())
ENC = {"A": 0, "C": 1, "G": 2, "T": 3}
WIN = {}
for h in MWJ["hits"]:
    WIN.setdefault(h["region"], []).append(h)


def build(r, HS):
    # verbatim from Pooled Regions 2: SNV x hit rows and motifbreakR calls on the hits HS of region r
    S_ = f"{POOL}/regions/r{r}"
    hits = pd.read_csv(f"{S_}/hits.csv").set_index("hit_id")
    snv = pd.read_csv(f"{S_}/snvs.csv")
    q = pd.read_csv(f"{S_}/snv_hit_dpwm.csv")
    q = q[q["hit_id"].isin(HS)].sort_values(["snv", "hit_id"]).reset_index(drop=True)
    sn = q["snv"].values
    st = np.r_[0, np.flatnonzero(sn[1:] != sn[:-1]) + 1]
    hi = np.searchsorted(HS, q["hit_id"].values)
    n_snv = len(st)
    MOT = hits.loc[HS, "motif"].values
    base = {}
    for h in WIN[r]:
        for i, b in enumerate(h["window"]):
            base.setdefault(h["start"] - C["motif_offset"] + i, b)
    lo = min(base)
    seq = np.array([ENC.get(base.get(x, "N"), 4) for x in range(lo, max(base) + 1)])
    s_ = snv.loc[sn[st]]
    pos, ref, alt = s_["local"].values - lo, s_["ref"].map(ENC).values, s_["alt"].map(ENC).values
    assert (seq[pos] == ref).all(), "rebuilt sequence disagrees with the SNV reference bases"
    out = []
    for m in sorted(set(MOT)):
        if m not in MBR_MATS:
            continue
        S, Sr, smin, smax = MBR_MATS[m]
        w = len(S)
        n = len(seq) - w + 1
        idx = np.arange(n)[:, None] + np.arange(w)[None, :]
        f_fwd, f_rev = S[np.arange(w), seq[idx]].sum(1), Sr[np.arange(w), seq[idx]].sum(1)
        stw = pos[:, None] - np.arange(w)[None, :]
        k = np.broadcast_to(np.arange(w), stw.shape)
        ok = (stw >= 0) & (stw < n)
        stc = np.clip(stw, 0, n - 1)
        ref_f, ref_r = np.where(ok, f_fwd[stc], -np.inf), np.where(ok, f_rev[stc], -np.inf)
        alt_f = ref_f + np.where(ok, S[k, alt[:, None]] - S[k, ref[:, None]], 0)
        alt_r = ref_r + np.where(ok, Sr[k, alt[:, None]] - Sr[k, ref[:, None]], 0)
        sr, sa = np.maximum(ref_f, ref_r).max(1), np.maximum(alt_f, alt_r).max(1)
        pr, pa = (sr - smin) / (smax - smin), (sa - smin) / (smax - smin)
        diff = sa - sr
        keep = np.isfinite(sr) & ((pr > MBR_THRESH) | (pa > MBR_THRESH)) & (np.abs(diff) >= MBR_NEUT)
        out += [(i, m, diff[i]) for i in np.flatnonzero(keep)]
    t = pd.DataFrame(out, columns=["i", "motif", "alleleDiff"])
    t["j"] = np.arange(len(t))
    rows_ = pd.DataFrame({"i": np.repeat(np.arange(n_snv), np.diff(np.r_[st, len(sn)])), "motif": MOT[hi], "h": hi})
    link = t[["j", "i", "motif"]].merge(rows_, on=["i", "motif"])
    keep = np.sort(link["j"].unique())
    t = t.loc[keep].reset_index(drop=True)
    return dict(snv=snv, n_snv=n_snv, snv_id=sn[st], y=snv.loc[sn[st], "log2FC"].values,
                DIFF=t["alleleDiff"].values, TI=t["i"].values, MOTP=t["motif"].values)


KP = pd.read_csv(f"{WORK_DIR}/kircher_deltaP/pairs.csv")
KB = (KP.assign(a=KP["dP"].abs()).sort_values("a", ascending=False, kind="stable").drop_duplicates(["region", "snv", "motif"])
      [["region", "snv", "motif", "dP"]].merge(KP.groupby(["region", "snv", "motif"])["P_ref"].max().reset_index(),
                                               on=["region", "snv", "motif"]))
HW = [h for h in MWJ["hits"] if h["motif"] in AUC]


def kircher_tier(thr):
    regs, calls = {}, []
    for r in sorted(INFO):
        hs = np.array(sorted(h["hit_id"] for h in HW if h["region"] == r and AUC[h["motif"]] >= thr))
        if not len(hs):
            continue
        R = build(r, hs)
        if R["n_snv"] < 20:
            continue
        regs[r] = R
        calls.append(pd.DataFrame({"region": r, "i": R["TI"], "snv": R["snv_id"][R["TI"]], "motif": R["MOTP"], "ad": R["DIFF"]}))
    Cc = pd.concat(calls, ignore_index=True).merge(KB, on=["region", "snv", "motif"], how="left")
    miss = Cc["dP"].isna().mean()
    Cc = Cc[Cc["dP"].notna()].reset_index(drop=True)
    off = {r: o for r, o in zip(regs, np.cumsum([0] + [regs[r]["n_snv"] for r in regs])[:-1])}
    T = dict(ad=Cc["ad"].values, tf=pd.factorize(Cc["motif"])[0], grp=(Cc["region"].map(off) + Cc["i"]).values,
             dP=Cc["dP"].values, pref=Cc["P_ref"].values)
    return regs, off, T, miss


KREG = []
for name, thr in K_TIERS:
    regs, off, T, miss = kircher_tier(thr)
    n = sum(R["n_snv"] for R in regs.values())
    A = ablation(T, n)
    for r, R in regs.items():
        sl = slice(off[r], off[r] + R["n_snv"])
        row = {"tier": name, "region": r, "cell": INFO[r]["cell"], "home": set(INFO[r]["cell"].split(",")) <= HOME,
               "motifbreakR": pear(A["motifbreakR"][sl], R["y"])}
        row.update({v: pear(A[v][sl], R["y"]) for v in VARIANTS})
        KREG.append(row)
KA = pd.DataFrame(KREG)
KA.to_csv(f"{OUT}/kircher_ablation_regions.csv", index=False)
summ = []
for (name, home), g in KA.groupby(["tier", "home"], sort=False):
    row = {"tier": name, "regions": f"{'HepG2/K562' if home else 'other cells'} ({len(g)})", "motifbreakR": g["motifbreakR"].mean()}
    row.update({v: g[v].mean() - g["motifbreakR"].mean() for v in VARIANTS})
    summ.append(row)
KAS = pd.DataFrame(summ).set_index(["tier", "regions"])
print("mean over regions: difference in Pearson from motifbreakR (first column: motifbreakR itself)\n")
print(KAS.T.to_string(float_format=fmt))

## 2.2 HepG2/K562 regions: leave one region out, and without regions 0 and 8

In [ ]:
rows = []
for name, g in KA[KA["home"]].groupby("tier", sort=False):
    g = g.set_index("region")
    for gname, v in GATES.items():
        d = g[v] - g["motifbreakR"]
        row = {"tier": name, "gate": gname, "all 4 regions": d.mean()}
        for r in g.index:
            row[f"without {r}"] = d.drop(r).mean()
        row["without 0 and 8"] = d.drop([r for r in (0, 8) if r in d.index]).mean()
        row["per region"] = "  ".join(f"{r}: {x:+.3f}" for r, x in d.items())
        rows.append(row)
KL = pd.DataFrame(rows).set_index(["tier", "gate"])
KL.to_csv(f"{OUT}/kircher_leave_one_region_out.csv")
print("mean gain over motifbreakR in the HepG2/K562 regions\n")
print(KL.to_string(float_format=fmt))

## End

Outputs in `deltaP_gate2/`: `malinois_ablation.csv`, `malinois_leave_one_tf_out.csv`, `kircher_ablation_regions.csv`,
`kircher_leave_one_region_out.csv`.